In [0]:
from pyspark.sql.functions import *

In [0]:
silver_table = "retailnova.silver.products"
gold_table = "retailnova.gold.dim_product"

In [0]:
df_src = spark.table(silver_table)

In [0]:
# Remove duplicate records
df_src = df_src.dropDuplicates(["product_id", "updated_at"])

In [0]:
from delta.tables import DeltaTable

In [0]:
# Find latest update for each product
latest_time = df_src.groupBy("product_id").agg(
    max("updated_at").alias("updated_at")
)

# Keep only latest record of each product
df_latest = df_src.join(
    latest_time,
    on=["product_id", "updated_at"],
    how="inner"
)

print("Latest products:", df_latest.count())


# Merge into Gold
delta_table = DeltaTable.forName(spark, gold_table)

delta_table.alias("target").merge(
    df_latest.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdate(
    set={
        "product_name": "source.product_name",
        "category": "source.category",
        "subcategory": "source.subcategory",
        "brand": "source.brand",
        "unit_price": "source.unit_price",
        "cost_price": "source.cost_price",
        "supplier_id": "source.supplier_id",
        "updated_at": "source.updated_at"
    }
).whenNotMatchedInsert(
    values={
        "product_id": "source.product_id",
        "product_name": "source.product_name",
        "category": "source.category",
        "subcategory": "source.subcategory",
        "brand": "source.brand",
        "unit_price": "source.unit_price",
        "cost_price": "source.cost_price",
        "supplier_id": "source.supplier_id",
        "updated_at": "source.updated_at"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.sql(f"""
        SELECT *
        FROM {gold_table}
        ORDER BY product_key
    """)
)

product_key,product_id,product_name,category,subcategory,brand,unit_price,cost_price,supplier_id,updated_at
1,P000030,Product 000030,Toys,Plush,Brand_094,1961.22,993.55,S00264,2026-07-09T00:00:00.000Z
2,P000038,Product 000038,Toys,Games,Brand_032,968.61,652.01,S00195,2026-07-25T00:00:00.000Z
3,P000050,Product 000050,Sports,null,Brand_046,1075.52,716.93,S00112,2026-08-16T00:00:00.000Z
4,P000098,Product 000098,Home Appliances,Sportswear,Brand_031,1244.8,791.8,S00350,2026-08-13T00:00:00.000Z
5,P000100,Product 000100,Fashion,Accessories,Brand_184,1084.98,836.49,S00361,2026-08-16T00:00:00.000Z
6,P000107,Product 000107,Toys,Games,Brand_038,349.35,284.7,S00090,2026-08-02T00:00:00.000Z
7,P000110,Product 000110,Beauty,Makeup,Brand_140,1505.5,1119.91,S00468,2026-07-12T00:00:00.000Z
8,P000115,Product 000115,Electronics,Accessories,Brand_164,254.52,195.89,S00035,2026-07-26T00:00:00.000Z
9,P000141,Product 000141,Home Appliances,Cooling,Brand_182,1834.87,895.09,S00150,2026-08-23T00:00:00.000Z
10,P000164,Product 000164,Books,Children,Brand_175,256.27,124.8,S00180,2026-07-11T00:00:00.000Z


In [0]:
%sql
SELECT COUNT(*) FROM retailnova.bronze.products;



COUNT(*)
10000


In [0]:
%sql
SELECT COUNT(*) FROM retailnova.silver.products;

COUNT(*)
10000


In [0]:
%sql
SELECT COUNT(*) FROM retailnova.gold.dim_product;

COUNT(*)
10000
